# MNIST + Fashion-MNIST: 20 Specialists (ResNet18) -> 1 Generalist (ResNet50)

1. **Specialists** – 20 small ResNet18 models. Specialist `d` answers "Is this object class `d`, or NOT `d`?" It has **one head with two outputs**: false and true. Each input is one 28x28 object randomly placed on a 64x64 canvas.
2. **Generalist** – one ResNet50 with **20 outputs**, one binary presence task per class. It sees a mixed canvas with **1, 2, or 3 objects**. Every object can be a digit or a Fashion-MNIST item.
3. **Transfer learning** – keep each isolated object on its own canvas, at the same position as in the composite. The specialists predict each component; their maximum score per class teaches the generalist. Only the composite is needed at inference.

The generalist learns from **soft teacher labels**, following the teacher-only training flow below. Repeated objects of the same class activate one label; this predicts classes present, not object counts or positions.

Overlap is limited to **40% of each 28x28 bounding box**, measured against the union of its neighbours. The optional last section explores Pareto trade-offs across the 20 generalist tasks.

## 0. Install & imports

In [ ]:
# Uncomment if you do not have the libraries yet:
# %pip install torch torchvision matplotlib numpy

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

## 1. Settings

Change these numbers to make training faster (smaller) or better (bigger).
If you have no GPU, keep the numbers small first!

In [ ]:
# Use GPU if available (much faster), otherwise CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():   # Apple Silicon
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)

In [ ]:
SEED = 42

BATCH_SIZE = 128
CANVAS_SIZE = 64
OBJECT_SIZE = 28
MAX_OBJECTS = 3
MAX_OVERLAP = 0.40
NUM_CLASSES = 20
THRES_RELATIVE = 0.5

CLASS_NAMES = [f"digit {d}" for d in range(10)] + [
    "fashion T-shirt/top", "fashion Trouser", "fashion Pullover", "fashion Dress",
    "fashion Coat", "fashion Sandal", "fashion Shirt", "fashion Sneaker",
    "fashion Bag", "fashion Ankle boot",
]

# --- Generalist ---
GEN_EPOCHS = 3
STACK_TRAIN_SAMPLES = 30000
STACK_VAL_SAMPLES = 2000
STACK_TEST_SAMPLES = 5000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# --- Specialists ---
SPEC_EPOCHS = 3            # how many passes over the data for each specialist
SPEC_TRAIN_SIZE = 20000    # number of single objects per specialist (None = all training objects)

## 2. Load MNIST and Fashion-MNIST

Classes 0–9 are digits; classes 10–19 are fashion items. Both use pixel values in [0, 1].
Hold out 10% of source training images for validation **before making composites**.
The official test images are used only for testing.

In [ ]:
transform = transforms.ToTensor()  # use the same pixel scale for both datasets

class MixedMNIST(Dataset):
    def __init__(self, train):
        self.mnist = datasets.MNIST("./data", train=train, download=True, transform=transform)
        self.fashion = datasets.FashionMNIST("./data", train=train, download=True, transform=transform)
        self.targets = np.concatenate([self.mnist.targets.numpy(), self.fashion.targets.numpy() + 10])

    def __len__(self):
        return len(self.mnist) + len(self.fashion)

    def __getitem__(self, idx):
        if idx < len(self.mnist):
            return self.mnist[idx]
        image, label = self.fashion[idx - len(self.mnist)]
        return image, label + 10

all_train = MixedMNIST(train=True)
test_full = MixedMNIST(train=False)

rng = np.random.RandomState(SEED)
indices = rng.permutation(len(all_train))
split = int(0.9 * len(indices))
train_ids, val_ids = indices[:split], indices[split:]
train_full = Subset(all_train, train_ids.tolist())
val_full = Subset(all_train, val_ids.tolist())
train_labels = all_train.targets[train_ids]

print("Train:", len(train_full), "| Validation:", len(val_full), "| Test:", len(test_full))

# Look at a few examples
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    img, label = train_full[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis("off")
plt.show()

## 3. Build the models

We modify ResNet a little because MNIST images are tiny and have **1 colour channel** (grey), not 3:
- `conv1`: accept 1 channel and use a small 3x3 filter (the original 7x7 is too big for 28x28 images).
- `fc` (the last layer = the "head"): set how many outputs we want.

In [ ]:
def make_resnet18(num_outputs=2):
    # Specialist: ResNet18 with a 2-output head -> [not_d, d]
    model = models.resnet18(weights=None)   # weights=None -> start from random weights
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.fc = nn.Linear(model.fc.in_features, num_outputs)
    return model

In [ ]:
def make_resnet50(num_outputs=NUM_CLASSES):
    # Generalist: ResNet50 with a 20-output head -> one binary presence task per class
    model = models.resnet50(weights=None)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.fc = nn.Linear(model.fc.in_features, num_outputs)
    return model

## 4. Specialist datasets ("class d" vs "not d")

Put one object at a random location on a 64x64 canvas. Positions are fixed by a seed.
For each specialist, label **1** means its own class and **0** means any of the other 19 classes.
All specialists see both digit and fashion examples.

In [ ]:
class SingleMNIST(Dataset):
    # One object, scattered around the same canvas size used by the generalist.
    def __init__(self, base_dataset, seed=SEED):
        self.base = base_dataset
        rng = np.random.RandomState(seed)
        self.positions = rng.randint(0, CANVAS_SIZE - OBJECT_SIZE + 1, (len(self.base), 2))

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        y, x = self.positions[idx]
        canvas = torch.zeros(1, CANVAS_SIZE, CANVAS_SIZE)
        canvas[:, y:y+OBJECT_SIZE, x:x+OBJECT_SIZE] = img
        return canvas, label

class BinaryMNIST(Dataset):
    # Changes the label to: 1 if label == digit (class ID), else 0.
    def __init__(self, base_dataset, digit):
        self.base = base_dataset
        self.digit = digit

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        return img, int(label == self.digit)

In [ ]:
# Use a random subset of the training data (faster). Set SPEC_TRAIN_SIZE=None for everything.
if SPEC_TRAIN_SIZE is None:
    spec_train_base = train_full
    spec_labels = train_labels
else:
    idx = np.random.permutation(len(train_full))[:SPEC_TRAIN_SIZE]
    spec_train_base = Subset(train_full, idx.tolist())
    spec_labels = train_labels[idx]

spec_train_base = SingleMNIST(spec_train_base, seed=SEED)
spec_test_base = SingleMNIST(test_full, seed=SEED + 1)
specialist_weights = []

## 5. Train the 20 specialists

Only about 5% of examples belong to a given class. Use the actual negative/positive ratio
as the positive-class loss weight, so always answering "false" is not rewarded.

In [ ]:
def train_specialist(digit):
    model = make_resnet18(num_outputs=2).to(DEVICE)

    train_loader = DataLoader(BinaryMNIST(spec_train_base, digit),
                              batch_size=BATCH_SIZE, shuffle=True)

    positives = int((spec_labels == digit).sum())
    negatives = len(spec_labels) - positives
    if positives == 0 or negatives == 0:
        raise ValueError("Each specialist needs positive and negative training examples.")
    pos_weight = negatives / positives
    specialist_weights.append(pos_weight)
    class_weights = torch.tensor([1.0, pos_weight]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(SPEC_EPOCHS):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()              # 1) clear old gradients
            outputs = model(images)            # 2) forward pass
            loss = criterion(outputs, labels)  # 3) how wrong are we?
            loss.backward()                    # 4) compute gradients
            optimizer.step()                   # 5) update weights

            total_loss += loss.item()
        print(f"  specialist {digit} | epoch {epoch+1}/{SPEC_EPOCHS} | loss {total_loss/len(train_loader):.4f}")

    model.eval()
    torch.save(model.state_dict(), f"mixed_specialist_{digit}.pt")
    return model

In [ ]:
specialists = []
specialist_weights = []
for d in range(NUM_CLASSES):
    print(f"=== Training specialist for {CLASS_NAMES[d]} ===")
    specialists.append(train_specialist(d))

# Needed to undo the class-weight shift when producing teacher probabilities.
torch.save(specialist_weights, "mixed_specialist_weights.pt")

## 6. Check the specialists on the test set

- **accuracy**: overall correct answers
- **recall on class d**: out of all real class-`d` objects, how many did the specialist find?

In [ ]:
@torch.no_grad()
def evaluate_specialist(model, digit):
    loader = DataLoader(BinaryMNIST(spec_test_base, digit), batch_size=256)
    correct, total, pos_correct, pos_total = 0, 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()
        pos_mask = labels == 1
        pos_correct += (preds[pos_mask] == 1).sum().item()
        pos_total += pos_mask.sum().item()
    return correct / total, pos_correct / max(pos_total, 1)

In [ ]:
for d, m in enumerate(specialists):
    acc, recall = evaluate_specialist(m, d)
    print(f"Specialist {CLASS_NAMES[d]}: accuracy {acc*100:.2f}% | recall: {recall*100:.2f}%")

### Bonus sanity check: do the 20 specialists work as a team?

Pick the class with the highest specialist probability on a single-object canvas.
Before taking softmax, undo the approximate log-odds shift introduced by weighted training.
This simple prior correction does not guarantee perfectly calibrated probabilities.

In [ ]:
@torch.no_grad()
def specialists_predict(images):
    # images: (B, 1, 64, 64) -> probabilities: (B, 20)
    probs = []
    for d, m in enumerate(specialists):
        m.eval()
        logits = m(images)
        logits[:, 1] -= np.log(specialist_weights[d])
        p = F.softmax(logits, dim=1)[:, 1]   # probability of class d
        probs.append(p)
    return torch.stack(probs, dim=1)

In [ ]:
loader = DataLoader(spec_test_base, batch_size=256)
correct = 0
for images, labels in loader:
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    correct += (specialists_predict(images).argmax(dim=1) == labels).sum().item()
print(f"Team of specialists, 20-class accuracy: {correct/len(spec_test_base)*100:.2f}%")

## 7. Stacked mixed dataset (for the generalist)

Each 64x64 image contains **1–3 objects**, sampled with equal probability. Each object can be a digit
or fashion item, including repeated classes. The 20-value multi-hot label records which classes are present.

Placement is random, with at most **40% of each object's box shared with the union of its neighbours**.
If random placement fails, use non-overlapping corners. Combine pixels with `maximum`, as before.
Keep each isolated component at its original position for the teachers. Padded unused slots have label `-1`.

In [ ]:
def overlap_fraction(positions):
    # Count the union of overlapping box pixels, not each neighbour separately.
    fractions = []
    for i, (y, x) in enumerate(positions):
        covered = np.zeros((OBJECT_SIZE, OBJECT_SIZE), dtype=bool)
        for j, (other_y, other_x) in enumerate(positions):
            if i == j:
                continue
            y0, y1 = max(y, other_y) - y, min(y, other_y) + OBJECT_SIZE - y
            x0, x1 = max(x, other_x) - x, min(x, other_x) + OBJECT_SIZE - x
            if y1 > y0 and x1 > x0:
                covered[y0:y1, x0:x1] = True
        fractions.append(covered.mean())
    return max(fractions)

class StackedMNIST(Dataset):
    def __init__(self, base_dataset, n_samples, seed):
        self.base = base_dataset
        rng = np.random.RandomState(seed)
        if not 1 <= MAX_OBJECTS <= 3 or CANVAS_SIZE < 2 * OBJECT_SIZE:
            raise ValueError("Use 1–3 objects and a canvas at least twice the object size.")
        self.counts = rng.randint(1, MAX_OBJECTS + 1, n_samples)
        self.indices = rng.randint(0, len(base_dataset), (n_samples, MAX_OBJECTS))
        self.positions = np.zeros((n_samples, MAX_OBJECTS, 2), dtype=np.int64)
        for i, count in enumerate(self.counts):
            for attempt in range(200):
                positions = rng.randint(0, CANVAS_SIZE - OBJECT_SIZE + 1, (count, 2))
                if overlap_fraction(positions) <= MAX_OVERLAP:
                    break
            else:
                corners = np.array([[0, 0], [0, CANVAS_SIZE - OBJECT_SIZE], [CANVAS_SIZE - OBJECT_SIZE, 0]])
                positions = corners[rng.permutation(3)[:count]]
            self.positions[i, :count] = positions

    def __len__(self):
        return len(self.counts)

    def __getitem__(self, i):
        components = torch.zeros(MAX_OBJECTS, 1, CANVAS_SIZE, CANVAS_SIZE)
        labels = torch.full((MAX_OBJECTS,), -1, dtype=torch.long)
        target = torch.zeros(NUM_CLASSES)
        for k in range(self.counts[i]):
            image, label = self.base[int(self.indices[i, k])]
            y, x = self.positions[i, k]
            components[k, :, y:y+OBJECT_SIZE, x:x+OBJECT_SIZE] = image
            labels[k] = label
            target[label] = 1.0

        stacked = components.max(dim=0).values
        active = labels >= 0
        return stacked, target, labels, components, active

In [ ]:
stack_train = StackedMNIST(train_full, STACK_TRAIN_SAMPLES, seed=1)
stack_val   = StackedMNIST(val_full,   STACK_VAL_SAMPLES,   seed=2)
stack_test  = StackedMNIST(test_full,  STACK_TEST_SAMPLES,  seed=3)

In [ ]:
# Look at random examples (re-run the cell to see new ones)
idxs = random.sample(range(len(stack_train)), 10)

fig, axes = plt.subplots(1, 10, figsize=(14, 2.5))
for idx, ax in zip(idxs, axes):
    x, target, labels, components, active = stack_train[idx]
    ax.imshow(x.squeeze(), cmap="gray")
    ax.set_title("\n".join(CLASS_NAMES[d] for d in labels[active].tolist()), fontsize=8)
    ax.axis("off")
plt.show()

## 8. Transfer learning: specialists teach the generalist

1. Ask all 20 specialists about each **isolated component canvas** at its original position.
2. Take the maximum probability for each class across the active components; skip padded empty slots.
3. Train the generalist on the full composite using these **soft targets**, without thresholding the teachers.

The main training loss uses teacher labels only. The true composite labels are used for evaluation.
The generalist has 20 binary tasks; each output answers whether its class is present anywhere.

In [ ]:
@torch.no_grad()
def teacher_targets(components, active):
    # components: (B, MAX_OBJECTS, 1, 64, 64); active: (B, MAX_OBJECTS)
    components, active = components.to(DEVICE), active.to(DEVICE)
    probs = torch.zeros(*active.shape, NUM_CLASSES, device=DEVICE)
    probs[active] = specialists_predict(components[active])
    return probs.max(dim=1).values   # class present if ANY real component contains it

In [ ]:
generalist = make_resnet50(num_outputs=NUM_CLASSES).to(DEVICE)

train_loader = DataLoader(stack_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(stack_val,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(stack_test,  batch_size=BATCH_SIZE)

# Keep the loss for each of the 20 binary tasks before combining them.
bce = nn.BCEWithLogitsLoss(reduction="none")

## 9. Train the generalist

In [ ]:
def train_generalist(model, saveFile, pref=None, method="ls"):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    if pref is None:
        pref = [1.0 / NUM_CLASSES] * NUM_CLASSES
    weights = torch.tensor(pref, dtype=torch.float32, device=DEVICE)
    weights = weights / weights.sum()
    famo = FAMO(NUM_CLASSES, pref) if method == "famo" else None

    for epoch in range(GEN_EPOCHS):
        model.train()
        run_soft = 0.0
        for images, targets, _, components, active in train_loader:
            images = images.to(DEVICE)
            soft = teacher_targets(components, active)
            logits = model(images)
            losses = bce(logits, soft).mean(dim=0)   # one loss per class
            loss = (weights * losses).sum()

            optimizer.zero_grad()
            if method == "ls":
                loss.backward()
            else:
                multi_task_backward(model, losses, weights, method, famo)
            optimizer.step()

            if famo is not None:
                # Recheck the same batch, without updating BatchNorm statistics twice.
                buffers = [(b, b.clone()) for b in model.buffers()]
                with torch.no_grad():
                    new_losses = bce(model(images), soft).mean(dim=0)
                    for buffer, saved in buffers:
                        buffer.copy_(saved)
                famo.update(losses.detach(), new_losses)

            run_soft += loss.item()

        print(f"Epoch {epoch+1}/{GEN_EPOCHS} | teacher loss {run_soft/len(train_loader):.4f}")

    torch.save(model.state_dict(), saveFile)

In [ ]:
GEN_EPOCHS = 7
train_generalist(model=generalist, saveFile="mixed_generalist_resnet50.pt")

## 10. Evaluate the generalist

Use the original **relative threshold**: a class is present when `probability > THRES_RELATIVE * max_probability`.
No fixed number of labels is forced. This rule can predict more than three classes; the **dataset** still has at most three objects.

- **Exact match**: predicted classes equal the true set, with nothing extra.
- **Per-class accuracy**: correctness of each binary task; absent labels can make this look high.

The other existing threshold functions remain available for comparison. The relative rule is used by default below and in the prediction plots.

In [ ]:
@torch.no_grad()
def evaluate_generalist_thresh_05(model, loader):
    model.eval()
    exact, total = 0, 0
    per_digit_correct = torch.zeros(NUM_CLASSES)
    for images, targets, *_ in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        preds = (torch.sigmoid(model(images)) > 0.5).float()

        exact += (preds == targets).all(dim=1).sum().item()
        per_digit_correct += (preds == targets).float().sum(dim=0).cpu()
        total += targets.size(0)

    print(f"Exact match (all classes found, nothing extra): {exact/total*100:.2f}%")
    for d in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[d]} yes/no accuracy: {per_digit_correct[d].item()/total*100:.2f}%")

In [ ]:
@torch.no_grad()
def evaluate_generalist_thresh_gap(model, loader):
    model.eval()
    exact, total = 0, 0
    per_digit_correct = torch.zeros(NUM_CLASSES)
    for images, targets, *_ in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)

        probs = torch.sigmoid(model(images))
        sorted_p, idx = probs.sort(dim=1, descending=True)
        gaps = sorted_p[:, :-1] - sorted_p[:, 1:]
        k = gaps.argmax(dim=1) + 1                      # digits to keep, per image
        rank = torch.arange(NUM_CLASSES, device=DEVICE).unsqueeze(0)
        keep = (rank < k.unsqueeze(1)).float()          # 1 for the top-k ranks
        preds = torch.zeros_like(probs).scatter_(1, idx, keep)

        exact += (preds == targets).all(dim=1).sum().item()
        per_digit_correct += (preds == targets).float().sum(dim=0).cpu()
        total += targets.size(0)

    print(f"Exact match (all classes found, nothing extra): {exact/total*100:.2f}%")
    for d in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[d]} yes/no accuracy: {per_digit_correct[d].item()/total*100:.2f}%")

In [ ]:
@torch.no_grad()
def evaluate_generalist_thresh_relative_max(model, loader, ratio=THRES_RELATIVE):
    model.eval()
    exact, total = 0, 0
    per_digit_correct = torch.zeros(NUM_CLASSES)
    for images, targets, *_ in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)

        probs = torch.sigmoid(model(images))                         # (batch, NUM_CLASSES)
        max_p = probs.max(dim=1, keepdim=True).values                # (batch, 1) max per image
        preds = (probs > ratio * max_p).float()                      # (batch, NUM_CLASSES) of 0/1

        exact += (preds == targets).all(dim=1).sum().item()
        per_digit_correct += (preds == targets).float().sum(dim=0).cpu()
        total += targets.size(0)

    print(f"Exact match: {exact/total*100:.2f}%")
    for d in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[d]} yes/no accuracy: {per_digit_correct[d].item()/total*100:.2f}%")

In [ ]:
@torch.no_grad()
def evaluate_generalist_thresh_relative_max_norm(model, loader, ratio=0.1):
    model.eval()
    exact, total = 0, 0
    per_digit_correct = torch.zeros(NUM_CLASSES)
    for images, targets, *_ in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)

        probs = torch.sigmoid(model(images))                         

        # 1. Hitung nilai min dan max untuk setiap gambar
        min_p = probs.min(dim=1, keepdim=True).values                
        max_p = probs.max(dim=1, keepdim=True).values                

        # 2. Normalisasi ke rentang [0, 1] (ditambah 1e-8 untuk menghindari pembagian dengan nol)
        norm_probs = (probs - min_p) / (max_p - min_p + 1e-8)        

        # 3. Terapkan threshold statis pada probabilitas yang sudah dinormalisasi
        preds = (norm_probs > ratio).float()                         

        exact += (preds == targets).all(dim=1).sum().item()
        per_digit_correct += (preds == targets).float().sum(dim=0).cpu()
        total += targets.size(0)

    print(f"Exact match: {exact/total*100:.2f}%")
    for d in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[d]} yes/no accuracy: {per_digit_correct[d].item()/total*100:.2f}%")

In [ ]:
# Optional comparison:
# evaluate_generalist_thresh_05(model=generalist, loader=test_loader)

In [ ]:
# Optional comparison:
# evaluate_generalist_thresh_gap(model=generalist, loader=test_loader)

In [ ]:
evaluate_generalist_thresh_relative_max(model=generalist, loader=test_loader)

In [ ]:
# Optional comparison:
# evaluate_generalist_thresh_relative_max_norm(model=generalist, loader=test_loader)

## 11. See predictions

In [ ]:
@torch.no_grad()
def show_predictions(model, n=8, ratio=THRES_RELATIVE):
    model.eval()
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 3))
    for ax in np.atleast_1d(axes):
        i = random.randrange(len(stack_test))
        x, target, labels, components, active = stack_test[i]     # composite: (1, 64, 64)

        probs = torch.sigmoid(model(x.unsqueeze(0).to(DEVICE)))[0].cpu()
        predicted = [d for d in range(NUM_CLASSES) if probs[d] > ratio * probs.max()]
        true = sorted(set(labels[active].tolist()))

        ax.imshow(x.squeeze(), cmap="gray")
        ax.set_title(f"true {[CLASS_NAMES[d] for d in true]}\npred {[CLASS_NAMES[d] for d in predicted]}", fontsize=8,
                     color="green" if predicted == true else "red")
        ax.axis("off")
    plt.show()

In [ ]:
show_predictions(model=generalist)

In [ ]:
show_predictions(model=generalist)

In [ ]:
show_predictions(model=generalist)

In [ ]:
show_predictions(model=generalist)

In [ ]:
@torch.no_grad()
def show_predictions_gap(model, n=8):
    model.eval()
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 3))
    for ax in np.atleast_1d(axes):
        i = random.randrange(len(stack_test))
        x, target, labels, components, active = stack_test[i]

        probs = torch.sigmoid(model(x.unsqueeze(0).to(DEVICE)))[0].cpu()   # (NUM_CLASSES,)

        sorted_p, idx = probs.sort(descending=True)
        gaps = sorted_p[:-1] - sorted_p[1:]
        k = gaps.argmax().item() + 1                  # number of digits to keep
        predicted = sorted(idx[:k].tolist())          # the top-k digits

        true = sorted(set(labels[active].tolist()))

        ax.imshow(x.squeeze(), cmap="gray")
        ax.set_title(f"true {true}\npred {predicted}",
                     color="green" if predicted == true else "red")
        ax.axis("off")
    plt.show()

In [ ]:
# Optional gap-threshold plot:
# show_predictions_gap(model=generalist)

In [ ]:
# Optional gap-threshold plot:
# show_predictions_gap(model=generalist)

In [ ]:
# Optional gap-threshold plot:
# show_predictions_gap(model=generalist)

In [ ]:
# Optional gap-threshold plot:
# show_predictions_gap(model=generalist)

In [ ]:
# Optional gap-threshold plot:
# show_predictions_gap(model=generalist)

## 12. Optional: generalist Pareto exploration

The 20 specialist models remain independent binary teachers. Here we vary the balance between the **20 generalist task losses** with LS, PCGrad, MGDA, or FAMO, using the methods from `okt-22/`.

The default training above is uniform LS. Set `RUN_PARETO=True` below to train additional candidates.
This is expensive: PCGrad/MGDA store one backbone gradient per class; reduce `BATCH_SIZE` or use LS/FAMO if memory is limited.
Candidates are compared on **all 20 validation BCE losses**, never test scores. A digit/fashion plot is just a two-dimensional summary.
Preference midpoints provide a simple expansion heuristic, not a guarantee of finding the complete Pareto front.

In [ ]:
def flat_grads(losses, params):
    gs = []
    for l in losses:
        g = torch.autograd.grad(l, params, retain_graph=True, allow_unused=True)
        gs.append(torch.cat([(gi if gi is not None else torch.zeros_like(p)).flatten() for gi, p in zip(g, params)]))
    return torch.stack(gs)                                   # (T, P)

def pcgrad(G):
    # project each task gradient away from conflicting ones (Yu et al. 2020)
    T = len(G); out = G.clone()
    for i in range(T):
        for j in random.sample(range(T), T):
            if j == i: continue
            dot = (out[i] * G[j]).sum()
            if dot < 0: out[i] = out[i] - dot / (G[j].norm() ** 2 + 1e-12) * G[j]
    return out.sum(0)

def mgda_weights(G, iters=50):
    # min-norm point in the convex hull of task gradients (Desideri 2012; Sener & Koltun 2018), Frank-Wolfe solver
    M = G @ G.T; T = len(G); a = torch.ones(T, device=G.device) / T
    for _ in range(iters):
        i = torch.argmin(M @ a); e = torch.zeros_like(a); e[i] = 1; d = e - a
        gamma = torch.clamp(-(d @ M @ a) / (d @ M @ d + 1e-12), 0, 1)
        a = a + gamma * d
    return a

class FAMO:
    # Fast Adaptive Multitask Optimization (Liu et al. 2023): O(1) space/time, balances rate of loss decrease
    def __init__(self, T, pref=None, lr=0.025, gamma=1e-3):
        init = torch.log(torch.tensor(pref if pref is not None else [1.0 / T] * T, dtype=torch.float32))
        self.w = init.clone().to(DEVICE).requires_grad_(True)
        self.opt = torch.optim.Adam([self.w], lr=lr, weight_decay=gamma); self.eps = 1e-8
        self.min_loss = torch.zeros(T, device=DEVICE)
    def weighted(self, losses):
        D = losses - self.min_loss + self.eps; z = F.softmax(self.w, -1); c = (z / D).sum().detach()
        return (D.log() * z / c).sum()
    def update(self, prev, curr):
        delta = (prev - self.min_loss + self.eps).log() - (curr - self.min_loss + self.eps).log()
        with torch.enable_grad():
            d = torch.autograd.grad(F.softmax(self.w, -1), self.w, grad_outputs=delta.detach())[0]
        self.opt.zero_grad(); self.w.grad = d; self.opt.step()

def multi_task_backward(model, losses, weights, method, famo):
    if method == "famo":
        famo.weighted(losses).backward()
        return
    if method not in ("pcgrad", "mgda"):
        raise ValueError("Unknown method: " + method)

    # Keep the existing ResNet structure; fc contains the 20 binary task outputs.
    shared = [p for name, p in model.named_parameters() if not name.startswith("fc.")]
    weighted = list(weights * losses)
    G = flat_grads(weighted, shared)
    sum(weighted).backward()  # fc keeps its own task gradients
    combined = pcgrad(G) if method == "pcgrad" else (mgda_weights(G) @ G) * NUM_CLASSES
    offset = 0
    for p in shared:
        size = p.numel()
        p.grad = combined[offset:offset+size].view_as(p).clone()
        offset += size

In [ ]:
@torch.no_grad()
def generalist_val_losses(model):
    model.eval()
    total = torch.zeros(NUM_CLASSES, device=DEVICE)
    for images, targets, *_ in val_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        total += bce(model(images), targets).sum(dim=0)
    return (total / len(stack_val)).cpu().numpy()

def nondominated(errors):
    errors = np.array(errors)
    return np.array([not any(np.all(b <= a) and np.any(b < a) for b in errors) for a in errors])

def pareto_run(method, pref, index):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    model = make_resnet50(num_outputs=NUM_CLASSES).to(DEVICE)
    # Reset batch shuffling after model initialization, consistently for each candidate.
    torch.manual_seed(SEED)
    filename = f"mixed_pareto_{index}_{method}.pt"
    train_generalist(model, filename, pref=pref, method=method)
    errors = generalist_val_losses(model)
    print(method, "| mean validation BCE:", errors.mean().round(4))
    return dict(method=method, pref=pref, errors=errors, filename=filename)

In [ ]:
RUN_PARETO = False
METHODS = ["ls", "pcgrad", "mgda", "famo"]
EXPAND_ROUNDS = 1

if RUN_PARETO:
    # Uniform, digit-emphasized, fashion-emphasized; every vector still has 20 tasks.
    preferences = [[1.0 / NUM_CLASSES] * NUM_CLASSES,
                   [0.08] * 10 + [0.02] * 10,
                   [0.02] * 10 + [0.08] * 10]
    results = []
    for method in METHODS:
        for pref in preferences:
            results.append(pareto_run(method, pref, len(results)))

    for _ in range(EXPAND_ROUNDS):
        keep = nondominated([r["errors"] for r in results])
        front = [r for r, ok in zip(results, keep) if ok]
        for method in METHODS:
            candidates = [r for r in front if r["method"] == method]
            pairs = [(a, b) for i, a in enumerate(candidates) for b in candidates[i+1:]]
            pairs.sort(key=lambda pair: np.linalg.norm(np.array(pair[0]["pref"]) - pair[1]["pref"]))
            for a, b in pairs:
                pref = ((np.array(a["pref"]) + b["pref"]) / 2).tolist()
                if any(r["method"] == method and np.allclose(r["pref"], pref) for r in results):
                    continue
                results.append(pareto_run(method, pref, len(results)))
                break  # at most one extra candidate per method in this round

    keep = nondominated([r["errors"] for r in results])
    best = min((r for r, ok in zip(results, keep) if ok), key=lambda r: r["errors"].mean())
    for method in METHODS:
        selected = [r for r in results if r["method"] == method]
        plt.scatter([r["errors"][:10].mean() for r in selected],
                    [r["errors"][10:].mean() for r in selected], label=method)
    plt.xlabel("Mean digit validation BCE")
    plt.ylabel("Mean fashion validation BCE")
    plt.title("Projection of 20-task Pareto candidates")
    plt.legend()
    plt.show()

    generalist.load_state_dict(torch.load(best["filename"], map_location=DEVICE, weights_only=True))
    print("Selected on validation:", best["filename"])
    evaluate_generalist_thresh_relative_max(generalist, test_loader)
    show_predictions(generalist)